In [1]:
"""
Pseudobulk Differential Expression Pipeline — All Cell Types
==============================================================
Runs pseudobulk DE (pyDESeq2) for 10 hypothesis-driven cell types,
each with its own organized output folder.

Output structure:
  /Users/adiallo/Desktop/Thesis/Results/AIM_2_Results/Differential_analysis/
    ├── 01_Tumor/
    ├── 02_CD8_exhausted/
    ├── 03_CD8_intermediate/
    ├── 04_CD8_progenitor/
    ├── 05_CD8_all/
    ├── 06_CAF/
    ├── 07_DC1/
    ├── 08_DC2/
    ├── 09_Macrophage/
    └── 10_Treg/

Run from your samples directory:
    cd /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples
    python pseudobulk_deseq2_all.py
"""

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import sys
import warnings
from datetime import datetime
from sklearn.neighbors import NearestNeighbors
from scipy.spatial import KDTree
from scipy import sparse

warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
VISIUM_DIR  = "/Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples"
META_PATH   = "/Users/adiallo/Desktop/Thesis/Data_Documents/data_all.csv"
RESULTS_ROOT = "/Users/adiallo/Desktop/Thesis/Results/AIM_2_Results/Differential_analysis"

# ============================================================
# CELL TYPE DEFINITIONS
# Each entry: folder name, display name, list of C2L column substrings to match, threshold
# ============================================================
CELL_TYPES = [
    {
        "folder": "01_Tumor",
        "name": "Tumor",
        "keys": ["Tumor_"],  # matches all Tumor_ prefixed columns
        "threshold": 0.05,
    },
    {
        "folder": "02_CD8_exhausted",
        "name": "CD8+ Exhausted (CXCL13+)",
        "keys": ["cTNI14", "cTNI15", "cTNI16"],
        "threshold": 0.01,  # rare subtype, lower threshold
    },
    {
        "folder": "03_CD8_intermediate",
        "name": "CD8+ Intermediate (GZMK+)",
        "keys": ["cTNI11"],
        "threshold": 0.01,
    },
    {
        "folder": "04_CD8_progenitor",
        "name": "CD8+ Progenitor (IL7R+)",
        "keys": ["cTNI10", "cTNI12"],
        "threshold": 0.01,
    },
    {
        "folder": "05_CD8_all",
        "name": "CD8+ T-cells (all)",
        "keys": ["cTNI10", "cTNI11", "cTNI12", "cTNI13", "cTNI14", "cTNI15", "cTNI16"],
        "threshold": 0.02,
    },
    {
        "folder": "06_CAF",
        "name": "CAF",
        "keys": ["cS27", "cS28", "cS29", "cS30", "cS31"],
        "threshold": 0.05,
    },
    {
        "folder": "07_DC1",
        "name": "DC1",
        "keys": ["cM03"],
        "threshold": 0.01,
    },
    {
        "folder": "08_DC2",
        "name": "DC2",
        "keys": ["cM04"],
        "threshold": 0.01,
    },
    {
        "folder": "09_Macrophage",
        "name": "Macrophage",
        "keys": ["cM01", "cM02"],
        "threshold": 0.03,
    },
    {
        "folder": "10_Treg",
        "name": "Treg",
        "keys": ["cTNI08", "cTNI09"],
        "threshold": 0.01,
    },
]

# ============================================================
# PARAMETERS
# ============================================================
MIN_SPOTS      = 20
MIN_GENES_EXPR = 10

# ============================================================
# REGION DEFINITION — from Define_regions.ipynb
# ============================================================
def define_regions_spatial(adata, pixel_scale_100um):
    coords = adata.obsm["spatial"]
    tumor_labels  = ["cancer_cancer"]
    stroma_labels = ["benign_stroma", "benign_fat", "inter", "stroma"]

    histo = adata.obs["histology"].astype(str)
    is_tumor  = histo.isin(tumor_labels).values
    is_stroma = histo.isin(stroma_labels).values

    region_labels = np.array(["Other"] * adata.n_obs, dtype=object)

    if is_tumor.sum() == 0:
        region_labels[is_stroma] = "Stroma"
        return region_labels
    if is_stroma.sum() == 0:
        region_labels[is_tumor] = "CT"
        return region_labels

    tumor_coords = coords[is_tumor]
    stroma_tree  = KDTree(coords[is_stroma])
    dists_to_stroma, _ = stroma_tree.query(tumor_coords, k=1)

    neighbor_threshold = 1.5 * pixel_scale_100um
    boundary_subset_mask = dists_to_stroma <= neighbor_threshold
    tumor_indices = np.where(is_tumor)[0]
    true_boundary_indices = tumor_indices[boundary_subset_mask]

    if len(true_boundary_indices) == 0:
        region_labels[is_tumor] = "CT"
        region_labels[is_stroma] = "Stroma"
        return region_labels

    boundary_coords = coords[true_boundary_indices]
    boundary_tree   = KDTree(boundary_coords)
    dists_to_boundary, _ = boundary_tree.query(coords)

    im_width_pixels = 5 * pixel_scale_100um
    is_proximal = dists_to_boundary <= im_width_pixels

    region_labels[is_proximal & (is_tumor | is_stroma)] = "IM"
    region_labels[is_tumor & (~is_proximal)] = "CT"
    region_labels[is_stroma & (~is_proximal)] = "Stroma"

    return region_labels


def classify_mets(row):
    any_mets = str(row["any_mets"]).strip().upper() in ["T", "TRUE", "1"]
    dist     = str(row["Distant_Mets"]).strip().upper() in ["T", "TRUE", "1"]
    if not any_mets: return "No_Mets"
    if dist: return "Distant_Mets"
    return "LN_Mets"


def get_cell_type_proportion(ab, ct_keys):
    """
    Given the full C2L abundance DataFrame and a list of column key substrings,
    sum matching columns and return proportion per spot.
    """
    matching_cols = []
    for key in ct_keys:
        matching_cols.extend([c for c in ab.columns if key in c])
    # Deduplicate
    matching_cols = list(dict.fromkeys(matching_cols))

    if len(matching_cols) == 0:
        return None, matching_cols

    ct_abundance = ab[matching_cols].sum(axis=1)
    total_abundance = ab.sum(axis=1)
    proportion = ct_abundance / total_abundance.replace(0, np.nan)

    return proportion, matching_cols


def make_volcano(ax, res_df, title_str):
    """Draw a single volcano plot."""
    res = res_df.dropna(subset=["padj", "log2FoldChange"])
    if len(res) == 0:
        ax.set_title(f"{title_str}\n(no data)")
        return

    sig_up   = (res["padj"] < 0.05) & (res["log2FoldChange"] > 1)
    sig_down = (res["padj"] < 0.05) & (res["log2FoldChange"] < -1)
    ns       = ~(sig_up | sig_down)

    ax.scatter(res.loc[ns, "log2FoldChange"], -np.log10(res.loc[ns, "padj"]),
               c="gray", alpha=0.3, s=5)
    ax.scatter(res.loc[sig_up, "log2FoldChange"], -np.log10(res.loc[sig_up, "padj"]),
               c="#D62728", alpha=0.6, s=10, label=f"Up ({sig_up.sum()})")
    ax.scatter(res.loc[sig_down, "log2FoldChange"], -np.log10(res.loc[sig_down, "padj"]),
               c="#1F77B4", alpha=0.6, s=10, label=f"Down ({sig_down.sum()})")

    top_genes = res.head(10).index
    for gene in top_genes:
        if res.loc[gene, "padj"] < 0.05:
            ax.annotate(gene,
                        (res.loc[gene, "log2FoldChange"], -np.log10(res.loc[gene, "padj"])),
                        fontsize=5, alpha=0.8)

    ax.axhline(-np.log10(0.05), color="black", linestyle=":", alpha=0.3)
    ax.axvline(1, color="black", linestyle=":", alpha=0.3)
    ax.axvline(-1, color="black", linestyle=":", alpha=0.3)
    ax.set_xlabel("log2 Fold Change")
    ax.set_ylabel("-log10(padj)")
    ax.set_title(title_str)
    ax.legend(fontsize=8)


# ============================================================
# LOAD METADATA AND SAMPLES (once, shared across all cell types)
# ============================================================
print("=" * 70)
print("Pseudobulk DE Pipeline — 10 Cell Types")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Results root: {RESULTS_ROOT}")
print("=" * 70)

meta = pd.read_csv(META_PATH)
meta["deident"] = meta["deident"].astype(str)
meta["met_group"] = meta.apply(classify_mets, axis=1)

visium_files = sorted(
    f for f in os.listdir(VISIUM_DIR)
    if f.endswith(".h5ad") and f.replace(".h5ad", "").isdigit()
)
print(f"Found {len(visium_files)} sample files\n")

# ============================================================
# PRE-LOAD: Read all samples once, store what we need
# ============================================================
print("Pre-loading all samples...")
print("-" * 40)

sample_cache = {}

for i, f in enumerate(visium_files):
    sid = f.replace(".h5ad", "")
    print(f"  [{i+1}/{len(visium_files)}] {sid}...", end=" ")

    try:
        adata = sc.read_h5ad(os.path.join(VISIUM_DIR, f))

        if "histology" not in adata.obs.columns:
            print("SKIPPED (no histology)")
            continue

        # Get raw counts
        gene_names = adata.var_names.tolist()
        X = adata.X
        if sparse.issparse(X):
            X = X.toarray()

        # C2L abundance
        ab = adata.obsm["means_cell_abundance_w_sf"]

        # Regions
        coords = adata.obsm["spatial"]
        nbrs = NearestNeighbors(n_neighbors=2).fit(coords)
        distances, _ = nbrs.kneighbors(coords)
        pixel_scale = np.median(distances[:, 1])
        region_labels = define_regions_spatial(adata, pixel_scale)

        # Clinical
        row = meta.loc[meta["deident"] == sid]
        if row.shape[0] != 1:
            print("SKIPPED (metadata mismatch)")
            continue
        clin = row.iloc[0]

        # Library sizes for QC
        lib_sizes = np.asarray(X.sum(axis=1)).flatten()

        sample_cache[sid] = {
            "X": X,
            "gene_names": gene_names,
            "ab": ab,
            "region_labels": region_labels,
            "clin": clin,
            "lib_sizes": lib_sizes,
        }

        n_ct = (region_labels == "CT").sum()
        n_im = (region_labels == "IM").sum()
        n_str = (region_labels == "Stroma").sum()
        print(f"OK (CT={n_ct}, IM={n_im}, Stroma={n_str})")

    except Exception as e:
        print(f"FAILED: {e}")

print(f"\nCached {len(sample_cache)} samples\n")

# Determine shared genes across all samples
all_gene_lists = [s["gene_names"] for s in sample_cache.values()]
shared_genes = list(set(all_gene_lists[0]))
for gl in all_gene_lists[1:]:
    shared_genes = [g for g in shared_genes if g in gl]
shared_genes = sorted(shared_genes)
print(f"Shared genes across all samples: {len(shared_genes)}")

# Pre-compute gene indices per sample
for sid, data in sample_cache.items():
    gene_idx = [data["gene_names"].index(g) for g in shared_genes]
    data["gene_idx"] = gene_idx


# ============================================================
# MAIN LOOP: Run pseudobulk DE for each cell type
# ============================================================
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

overall_summary = []

for ct_info in CELL_TYPES:
    ct_folder = ct_info["folder"]
    ct_name   = ct_info["name"]
    ct_keys   = ct_info["keys"]
    ct_thresh = ct_info["threshold"]

    output_dir = os.path.join(RESULTS_ROOT, ct_folder)
    os.makedirs(output_dir, exist_ok=True)

    # Start log
    log_lines = []
    def log(msg):
        print(msg)
        log_lines.append(msg)

    log("\n" + "=" * 70)
    log(f"CELL TYPE: {ct_name} ({ct_folder})")
    log(f"  Keys: {ct_keys}")
    log(f"  Threshold: {ct_thresh:.0%}")
    log(f"  Output: {output_dir}")
    log("=" * 70)

    # --- STEP 1: Pseudobulk aggregation ---
    pseudobulk_counts = {}
    pseudobulk_meta = []

    for sid, data in sample_cache.items():
        ab = data["ab"]
        proportion, matched_cols = get_cell_type_proportion(ab, ct_keys)

        if proportion is None:
            log(f"  {sid}: SKIPPED (no matching columns)")
            continue

        X_shared = data["X"][:, data["gene_idx"]]
        region_labels = data["region_labels"]
        clin = data["clin"]

        regions_found = []
        for region in ["CT", "IM", "Stroma"]:
            mask = (region_labels == region) & (proportion.values > ct_thresh)
            n_spots = mask.sum()

            if n_spots < MIN_SPOTS:
                continue

            counts_sum = X_shared[mask, :].sum(axis=0)
            counts_sum = np.asarray(counts_sum).flatten()

            sample_key = f"{sid}_{region}"
            pseudobulk_counts[sample_key] = counts_sum

            mean_lib = np.log1p(np.mean(data["lib_sizes"][mask]))

            pseudobulk_meta.append({
                "sample_key": sample_key,
                "patient_id": sid,
                "region": region,
                "mets_status": classify_mets(clin),
                "MLH1": int(clin["MLH1"]),
                "age": int(clin["age"]),
                "sex": clin["sex"],
                "n_spots": n_spots,
                "QC": mean_lib,
            })
            regions_found.append(f"{region}({n_spots})")

    if len(pseudobulk_meta) == 0:
        log(f"  NO PSEUDOBULK SAMPLES — skipping {ct_name}")
        continue

    meta_df = pd.DataFrame(pseudobulk_meta).set_index("sample_key")
    count_df = pd.DataFrame(pseudobulk_counts, index=shared_genes).T
    count_df = count_df.loc[meta_df.index].astype(int)

    # Filter genes
    genes_detected = (count_df > 0).sum(axis=0)
    keep_genes = genes_detected[genes_detected >= MIN_GENES_EXPR].index
    count_df = count_df[keep_genes]

    log(f"\n  Pseudobulk samples: {count_df.shape[0]}")
    log(f"  Genes: {count_df.shape[1]}")
    log(f"  Region × Mets:")
    crosstab = pd.crosstab(meta_df["region"], meta_df["mets_status"])
    log(str(crosstab))

    # Save intermediates
    count_df.to_csv(os.path.join(output_dir, "pseudobulk_counts.csv"))
    meta_df.to_csv(os.path.join(output_dir, "pseudobulk_metadata.csv"))

    # --- STEP 2: pyDESeq2 ---
    meta_df["group"] = meta_df["region"] + "_" + meta_df["mets_status"]
    meta_df["group"] = pd.Categorical(meta_df["group"])
    meta_df["sex"] = pd.Categorical(meta_df["sex"])
    meta_df["MLH1"] = meta_df["MLH1"].astype(int)

    try:
        dds = DeseqDataSet(
            counts=count_df,
            metadata=meta_df,
            design="~ QC + MLH1 + age + sex + group"
        )
        dds.deseq2()
    except Exception as e:
        log(f"  DESeq2 FAILED: {e}")
        continue

    # --- STEP 3: Contrasts ---
    contrasts = {
        "IM_vs_CT__No_Mets": ["group", "IM_No_Mets", "CT_No_Mets"],
        "IM_vs_CT__LN_Mets": ["group", "IM_LN_Mets", "CT_LN_Mets"],
        "IM_vs_CT__Distant_Mets": ["group", "IM_Distant_Mets", "CT_Distant_Mets"],
        "Distant_vs_No__CT": ["group", "CT_Distant_Mets", "CT_No_Mets"],
        "Distant_vs_No__IM": ["group", "IM_Distant_Mets", "IM_No_Mets"],
        "Distant_vs_No__Stroma": ["group", "Stroma_Distant_Mets", "Stroma_No_Mets"],
        "LN_vs_No__CT": ["group", "CT_LN_Mets", "CT_No_Mets"],
        "LN_vs_No__IM": ["group", "IM_LN_Mets", "IM_No_Mets"],
        "LN_vs_No__Stroma": ["group", "Stroma_LN_Mets", "Stroma_No_Mets"],
        "Stroma_vs_CT__No_Mets": ["group", "Stroma_No_Mets", "CT_No_Mets"],
        "Stroma_vs_IM__No_Mets": ["group", "Stroma_No_Mets", "IM_No_Mets"],
    }

    results_all = {}
    summary_rows = []

    for cname, contrast in contrasts.items():
        if contrast[1] not in meta_df["group"].values or contrast[2] not in meta_df["group"].values:
            log(f"  SKIPPED {cname}: group missing from data")
            continue

        try:
            stat_res = DeseqStats(dds, contrast=contrast, alpha=0.05)
            stat_res.summary()
            res_df = stat_res.results_df.sort_values("padj")
            results_all[cname] = res_df

            n_sig  = (res_df["padj"] < 0.05).sum()
            n_up   = ((res_df["padj"] < 0.05) & (res_df["log2FoldChange"] > 0)).sum()
            n_down = ((res_df["padj"] < 0.05) & (res_df["log2FoldChange"] < 0)).sum()

            summary_rows.append({
                "cell_type": ct_name,
                "contrast": cname,
                "n_sig_005": n_sig,
                "n_up": n_up,
                "n_down": n_down,
                "top_gene": res_df.index[0] if n_sig > 0 else "none",
                "top_padj": res_df["padj"].iloc[0],
                "top_lfc": res_df["log2FoldChange"].iloc[0],
            })

            log(f"  {cname}: {n_sig} DE genes (↑{n_up} ↓{n_down})")
            res_df.to_csv(os.path.join(output_dir, f"DE_{cname}.csv"))

        except Exception as e:
            log(f"  FAILED {cname}: {e}")

    # --- STEP 4: Volcano plots ---
    volcano_groups = {
        "IM_vs_CT_by_mets": {
            "title": "IM vs CT — by Metastasis Status",
            "contrasts": ["IM_vs_CT__No_Mets", "IM_vs_CT__LN_Mets", "IM_vs_CT__Distant_Mets"],
        },
        "Distant_vs_No_by_region": {
            "title": "Distant Mets vs No Mets — by Region",
            "contrasts": ["Distant_vs_No__CT", "Distant_vs_No__IM", "Distant_vs_No__Stroma"],
        },
        "LN_vs_No_by_region": {
            "title": "LN Mets vs No Mets — by Region",
            "contrasts": ["LN_vs_No__CT", "LN_vs_No__IM", "LN_vs_No__Stroma"],
        },
        "Stroma_comparisons": {
            "title": "Stroma Comparisons — No Mets",
            "contrasts": ["Stroma_vs_CT__No_Mets", "Stroma_vs_IM__No_Mets"],
        },
    }

    for group_key, group_info in volcano_groups.items():
        contrast_list = [c for c in group_info["contrasts"] if c in results_all]
        if len(contrast_list) == 0:
            continue

        ncols = len(contrast_list)
        fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))
        fig.suptitle(f"{ct_name} — {group_info['title']}",
                     fontsize=13, fontweight="bold")

        if ncols == 1:
            axes = [axes]

        for ax, cn in zip(axes, contrast_list):
            display_name = cn.replace("__", " | ").replace("_", " ")
            make_volcano(ax, results_all[cn], display_name)

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"volcano_{group_key}.png"), dpi=150, bbox_inches="tight")
        plt.savefig(os.path.join(output_dir, f"volcano_{group_key}.pdf"), bbox_inches="tight")
        plt.close()

    # Save cell-type-level summary
    if summary_rows:
        ct_summary = pd.DataFrame(summary_rows)
        ct_summary.to_csv(os.path.join(output_dir, "DE_summary.csv"), index=False)
        overall_summary.extend(summary_rows)

    # Save log
    with open(os.path.join(output_dir, "run_log.txt"), "w") as f:
        f.write(f"Run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Cell type: {ct_name}\n")
        f.write(f"C2L keys: {ct_keys}\n")
        f.write(f"Threshold: {ct_thresh}\n")
        f.write(f"Min spots: {MIN_SPOTS}\n")
        f.write(f"Matched C2L columns: {matched_cols}\n\n")
        f.write("\n".join(log_lines))

    log(f"\n  ✓ {ct_name} complete — results in {ct_folder}/")

# ============================================================
# GLOBAL SUMMARY
# ============================================================
print("\n" + "=" * 70)
print("ALL CELL TYPES COMPLETE")
print("=" * 70)

if overall_summary:
    global_df = pd.DataFrame(overall_summary)
    global_df.to_csv(os.path.join(RESULTS_ROOT, "DE_summary_all_cell_types.csv"), index=False)

    print("\nGlobal DE Summary:")
    print(global_df[["cell_type", "contrast", "n_sig_005", "n_up", "n_down"]].to_string(index=False))

    # Summary heatmap: n_sig by cell type × contrast
    pivot = global_df.pivot_table(index="cell_type", columns="contrast",
                                   values="n_sig_005", fill_value=0)
    fig, ax = plt.subplots(figsize=(16, 8))
    im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=10)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, str(int(pivot.values[i, j])), ha="center", va="center",
                    fontsize=7, color="white" if pivot.values[i, j] > pivot.values.max() * 0.5 else "black")
    plt.colorbar(im, ax=ax, label="# DE genes (padj < 0.05)")
    ax.set_title("Pseudobulk DE Summary — All Cell Types × All Contrasts", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_ROOT, "DE_heatmap_all_cell_types.png"), dpi=150, bbox_inches="tight")
    plt.savefig(os.path.join(RESULTS_ROOT, "DE_heatmap_all_cell_types.pdf"), bbox_inches="tight")
    plt.close()
    print(f"\nHeatmap saved to {RESULTS_ROOT}/DE_heatmap_all_cell_types.pdf")

print(f"\nAll results in: {RESULTS_ROOT}/")
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Pseudobulk DE Pipeline — 10 Cell Types
Started: 2026-03-05 19:02:14
Results root: /Users/adiallo/Desktop/Thesis/Results/AIM_2_Results/Differential_analysis
Found 41 sample files

Pre-loading all samples...
----------------------------------------
  [1/41] 0... OK (CT=2901, IM=1308, Stroma=2080)
  [2/41] 10... OK (CT=3400, IM=585, Stroma=1787)
  [3/41] 100... OK (CT=6992, IM=168, Stroma=0)
  [4/41] 102... OK (CT=687, IM=0, Stroma=3681)
  [5/41] 106... OK (CT=4256, IM=1103, Stroma=94)
  [6/41] 107... OK (CT=3206, IM=366, Stroma=2908)
  [7/41] 11... OK (CT=4787, IM=903, Stroma=68)
  [8/41] 111... OK (CT=4913, IM=1073, Stroma=838)
  [9/41] 116... OK (CT=6106, IM=965, Stroma=83)
  [10/41] 117... OK (CT=4562, IM=126, Stroma=18)
  [11/41] 119... OK (CT=1322, IM=1003, Stroma=56)
  [12/41] 122... OK (CT=3826, IM=1002, Stroma=0)
  [13/41] 127... OK (CT=6426, IM=378, Stroma=90)
  [14/41] 128... OK (CT=5012, IM=0, Stroma=0)
  [15/41] 13... OK (CT=2949, IM=784, Stroma=226)
  [16/41] 14... OK (CT=27

Fitting size factors...
... done in 0.03 seconds.

/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:378: RuntimeWarning: overflow encountered in exp
  mu_ = np.maximum(size_factors * np.exp(X @ beta), min_mu)
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:232: RuntimeWarning: invalid value encountered in multiply
  - counts * np.log(mu)
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:230: RuntimeWarning: invalid value encountered in subtract
  -logbinom
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:382: RuntimeWarning: overflow encountered in exp
  mu_ = np.maximum(size_factors * np.exp(X @ beta), min_mu)
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:385: RuntimeWarning: invalid value encountered in divide
  + ((1 / disp + counts) * mu_ / (1 / disp + mu_)) @ X
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:378: RuntimeWarning: 

Log2 fold change & Wald test p-value: group IM_No_Mets vs CT_No_Mets
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1CF     198.431849       -0.522928  0.406490 -1.286448  0.198287  0.779631
A2M     5557.074047        0.882411  0.288967  3.053671  0.002261  0.104558
A4GALT   386.917320        0.576070  0.273626  2.105322  0.035263  0.413064
AAAS     708.829684        0.102265  0.122837  0.832521  0.405115  0.895785
AACS     539.237544       -0.199628  0.148010 -1.348747  0.177418  0.760310
...             ...             ...       ...       ...       ...       ...
ZWINT    613.617832       -0.150143  0.260166 -0.577106  0.563868  0.934594
ZXDC     455.724850       -0.252380  0.093596 -2.696471  0.007008  0.170626
ZYG11B   524.051554        0.029097  0.110286  0.263831  0.791910  0.976537
ZYX     3615.862699        0.212545  0.192335  1.105076  0.269127  0.829355
ZZEF1    641.275013        0.076652  0.132766  0.577343  0.563708  0.934594

[13598 rows x 6 co

... done in 0.95 seconds.

Running Wald tests...


Log2 fold change & Wald test p-value: group IM_LN_Mets vs CT_LN_Mets
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1CF     198.431849       -0.617779  0.490549 -1.259363  0.207899  0.590191
A2M     5557.074047        0.836754  0.347445  2.408307  0.016027  0.246915
A4GALT   386.917320        0.871848  0.328036  2.657778  0.007866  0.185250
AAAS     708.829684       -0.050366  0.147492 -0.341482  0.732741  0.893813
AACS     539.237544       -0.473755  0.178434 -2.655074  0.007929  0.185773
...             ...             ...       ...       ...       ...       ...
ZWINT    613.617832       -0.437525  0.312740 -1.399007  0.161811  0.546387
ZXDC     455.724850       -0.114494  0.112885 -1.014260  0.310459  0.665491
ZYG11B   524.051554       -0.031314  0.132383 -0.236542  0.813012  0.931265
ZYX     3615.862699       -0.049800  0.231199 -0.215397  0.829457  0.937706
ZZEF1    641.275013        0.154546  0.159526  0.968783  0.332653  0.681972

[13598 rows x 6 co

...

Log2 fold change & Wald test p-value: group Stroma_LN_Mets vs Stroma_No_Mets
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1CF     220.227899        0.416542  0.467291  0.891397  0.372716  0.999935
A2M     6859.777394        0.218023  0.313914  0.694531  0.487349  0.999935
A4GALT   462.817573        0.070382  0.298468  0.235811  0.813579  0.999935
AAAS     818.597432        0.092558  0.139984  0.661202  0.508483  0.999935
AACS     610.246369       -0.051823  0.170179 -0.304522  0.760730  0.999935
...             ...             ...       ...       ...       ...       ...
ZWINT    685.552484       -0.353041  0.287943 -1.226078  0.220169  0.999935
ZXDC     531.431541        0.115878  0.111735  1.037080  0.299698  0.999935
ZYG11B   598.431739        0.039175  0.123007  0.318481  0.750120  0.999935
ZYX     4181.817654       -0.063734  0.202960 -0.314023  0.753503  0.999935
ZZEF1    750.600233       -0.077854  0.149223 -0.521733  0.601857  0.999935

[13598 row

... done in 0.46 seconds.

Running Wald tests...


Log2 fold change & Wald test p-value: group Stroma_No_Mets vs CT_No_Mets
           baseMean  log2FoldChange     lfcSE      stat        pvalue  \
A1CF     220.227899       -1.509512  0.471535 -3.201276  1.368205e-03   
A2M     6859.777394        1.982544  0.322972  6.138441  8.333512e-10   
A4GALT   462.817573        1.391437  0.306625  4.537907  5.681540e-06   
AAAS     818.597432        0.062056  0.141152  0.439640  6.601979e-01   
AACS     610.246369       -0.783995  0.169802 -4.617106  3.891290e-06   
...             ...             ...       ...       ...           ...   
ZWINT    685.552484       -1.061907  0.290916 -3.650214  2.620221e-04   
ZXDC     531.431541       -0.502661  0.109323 -4.597952  4.266645e-06   
ZYG11B   598.431739        0.262515  0.123273  2.129546  3.320912e-02   
ZYX     4181.817654        0.700184  0.208517  3.357920  7.853127e-04   
ZZEF1    750.600233        0.280958  0.150821  1.862863  6.248156e-02   

                padj  
A1CF    5.462377e-03  
A2M 

... done in 0.36 seconds.



Log2 fold change & Wald test p-value: group Stroma_No_Mets vs IM_No_Mets
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1CF     220.227899       -0.826981  0.427730 -1.933419  0.053185  0.168189
A2M     6859.777394        0.851805  0.289416  2.943188  0.003249  0.030036
A4GALT   462.817573        0.618355  0.275371  2.245534  0.024734  0.105061
AAAS     818.597432        0.002260  0.128018  0.017655  0.985914  0.993884
AACS     610.246369       -0.475455  0.154307 -3.081231  0.002061  0.023283
...             ...             ...       ...       ...       ...       ...
ZWINT    685.552484       -0.774952  0.262199 -2.955584  0.003121  0.029400
ZXDC     531.431541       -0.233815  0.100709 -2.321702  0.020249  0.091779
ZYG11B   598.431739        0.220093  0.112487  1.956599  0.050395  0.162015
ZYX     4181.817654        0.406869  0.187038  2.175331  0.029605  0.117087
ZZEF1    750.600233        0.172570  0.136607  1.263263  0.206495  0.398242

[13598 rows x 